In [ ]:
import pickle
from datetime import datetime, timedelta, UTC
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs
import plotly.express as px
import psycopg
import psycopg_pool
from psycopg import sql

from aare.constants import TIME, TEMP
from aare_influx.field_request import FieldRequest
from aare_influx.remote_existenz_store import RemoteExistenzStore
from aare_timescale.postgres import copy_to_df
from aare_train.preparation import resample
from aare_train.utils import join_many, between

Turns out simulation at 00:00 probably fetches different past data from influx than simulation at 00:15. Need to investigate how much different and whether it has an impact.
For full reproducibility, would need to also archive the fetched target data in the postgres db (instead of only covariates).

Used this patch to get those files

```diff
diff --git a/src/oraku-forecast/main.py b/src/oraku-forecast/main.py
index 57614bd..56be7e4 100644
--- a/src/oraku-forecast/main.py
+++ b/src/oraku-forecast/main.py
@@ -150,6 +150,11 @@ async def forecast_once(
         )

         forecast["run_ts"] = run_ts
+
+        with open(f"./debugging/forecast-{run_ts.isoformat()}{'-sim' if use_cached_external else ''}.pkl",
+                  "wb") as file:
+            import pickle
+            pickle.dump(forecast, file)
     except Exception as e:
         # only catches errors during fetching and forecasting, mostly because fetching has external factors.
         # issues with the database or loading the model will only be visible in the app/container logs.
@@ -175,10 +180,21 @@ async def make_forecast(
     # configure and pull external sources
     sources = SourceRegistry.configure_sources(conn_pool)
     external_data = await load_external_data(sources, run_ts, use_cached_external)
+    with open(f"./debugging/external_data-{run_ts.isoformat()}{'-sim' if use_cached_external else ''}.pkl",
+              "wb") as file:
+        import pickle
+        pickle.dump(external_data, file)

     # compile inference data from internal (influx) and external data
     data = get_inference_data(model_meta["features"], model.extreme_lags, external_data, run_ts)
+    with open(f"./debugging/data-unscaled-{run_ts.isoformat()}{'-sim' if use_cached_external else ''}.pkl",
+              "wb") as file:
+        import pickle
+        pickle.dump(data, file)
     data = scale_inference_data(data, scalers)
+    with open(f"./debugging/data-scaled-{run_ts.isoformat()}{'-sim' if use_cached_external else ''}.pkl", "wb") as file:
+        import pickle
+        pickle.dump(data, file)

     # actually make forecast with loaded model
     return predict(model, data, scalers.get("series") if scalers else None, horizon, num_samples)
```

at ~17:00:03
just forecast

then more or less directly (17:04)
uv run src/oraku-forecast/main.py --simulate-runts 2026-05-03T15:00:12.595844+00:00

then 15min later, at ~17:15
uv run src/oraku-forecast/main.py --simulate-runts 2026-05-03T15:00:12.595844+00:00

then at ~17:34
uv run src/oraku-forecast/main.py --simulate-runts 2026-05-03T15:00:12.595844+00:00

```
│   ├── .
│   ├── actual
│   │   ├── data-scaled-2026-05-03T15:00:12.595844+00:00.pkl [7bd95bacc95b0f272c133c27c990b09cb2b3e85d5957d4c5bfff3a75bd1cd74d]
│   │   ├── data-unscaled-2026-05-03T15:00:12.595844+00:00.pkl [db07afa17d41f59675b569d0394490c3df2672cb5e62f3cf4778454e1694351d]
│   │   ├── external_data-2026-05-03T15:00:12.595844+00:00.pkl [472b36174e3a5188f137963a3c23000b0df121c1b78b7a63adf2fa85775c3c0c]
│   │   ├── forecast-2026-05-03T15:00:12.595844+00:00.pkl [cdce0c47d53869429c867064bbf3e592fdae2f34e160a94c4ced23a8bc9aa11e]
│   │   ├── log.txt [6a73cac4ec71254e2335f039550fa453b21d0b62e42b2d19657730e3cb9aa6b9]
│   ├── output_sim
│   │   ├── 2026-05-03T17:04:10_nowcasting_temp-1.0.parquet [51760f0ebd0611438f18b041954ca2281a75827df8a4f77f86432f1b51e08c06]
│   │   ├── 2026-05-03T17:15:57_nowcasting_temp-1.0.parquet [5a85e3964341d813143751782cebf5807463abfc138de6a9a74b0fb032acf3b9]
│   │   ├── 2026-05-03T17:35:04_nowcasting_temp-1.0.parquet [5a85e3964341d813143751782cebf5807463abfc138de6a9a74b0fb032acf3b9]
│   ├── sim_1704
│   │   ├── data-scaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [7bd95bacc95b0f272c133c27c990b09cb2b3e85d5957d4c5bfff3a75bd1cd74d]
│   │   ├── data-unscaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [db07afa17d41f59675b569d0394490c3df2672cb5e62f3cf4778454e1694351d]
│   │   ├── external_data-2026-05-03T15:00:12.595844+00:00-sim.pkl [d70328b82c594b03b1ddf0e7bdd1b38a68cfda9ac9bbd28b20772b799a98e9d6]
│   │   ├── forecast-2026-05-03T15:00:12.595844+00:00-sim.pkl [cdce0c47d53869429c867064bbf3e592fdae2f34e160a94c4ced23a8bc9aa11e]
│   │   ├── log.txt [561574390f8d90f39da688847e61d8bbe95e7ed608f77a036ac0a5f081b44c36]
│   ├── sim_1715
│   │   ├── data-scaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [abc023fa1857d134ecbe39215a9e2cc2a1117b7a98ad967e8167357416d610b4]
│   │   ├── data-unscaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [cf44488ce237986c7483a752f959286781fece6513d55533a0cf79f65b497957]
│   │   ├── external_data-2026-05-03T15:00:12.595844+00:00-sim.pkl [d70328b82c594b03b1ddf0e7bdd1b38a68cfda9ac9bbd28b20772b799a98e9d6]
│   │   ├── forecast-2026-05-03T15:00:12.595844+00:00-sim.pkl [b5c74f10104dbf922d7a1dd3a9db4f5b1153d5b53d572fbcacd463a8f7184d12]
│   │   ├── log.txt [ebe313eb0eab1ba26e23d3488e195e65590e71b076b133644d14f8a0a30db222]
│   ├── sim-1735
│   │   ├── data-scaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [abc023fa1857d134ecbe39215a9e2cc2a1117b7a98ad967e8167357416d610b4]
│   │   ├── data-unscaled-2026-05-03T15:00:12.595844+00:00-sim.pkl [cf44488ce237986c7483a752f959286781fece6513d55533a0cf79f65b497957]
│   │   ├── external_data-2026-05-03T15:00:12.595844+00:00-sim.pkl [d70328b82c594b03b1ddf0e7bdd1b38a68cfda9ac9bbd28b20772b799a98e9d6]
│   │   ├── forecast-2026-05-03T15:00:12.595844+00:00-sim.pkl [b5c74f10104dbf922d7a1dd3a9db4f5b1153d5b53d572fbcacd463a8f7184d12]
│   │   ├── log.txt [57a1b73580a0e3aa7ecd840240a2986f5eedef8e99b19749e942a7ff28346292]
```

Initial notes:

- External data changes between real and simulation, maybe because of float precision and storage? TODO confirm
- Direct and initial simulation have the same inference data (scaled and unscaled), despite different external data. Ultimately, this leads to the same forecast.
- Simulation 15min later has the same external data as initial simulation, supporting theory that it's about storage/retrieval, not some inherent fluctuation.
- Simulation 15min and 34min later have different inference data than the direct call and initial simulation. The external data is the same, so it's most likely that different data is fetched from influx??
- Logs confirm that influx query is the same every time.

TODO analyze pickles and check differences. It's possible that the influxdb is updated at 00:00:>12 but <00:15:00, but by how much and what's the impact?
TODO test with FIRST features, because it's very possible that the last 10min value hasn't reached influx db at 00:00:12 yet, so the MEAN agg is different, but the FIRST agg wouldn't be!


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
import logging

logging.basicConfig(level="DEBUG")
logging.getLogger("fsspec").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("dulwich").setLevel(logging.WARNING)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
exp_dir = Path("../debugging/2026-05-03T15:00:12.595844+00:00")
dir_actual = exp_dir / "actual"
dir_sim_04 = exp_dir / "sim_1704"
dir_sim_15 = exp_dir / "sim_1715"
# dir_sim_35 = exp_dir / "sim_1735"  not needed, identical to 1715

In [ ]:
def load(path: Path):
    if not path.exists():
        ls = list(path.parent.glob(path.name + "*.pkl"))
        if len(ls) != 1:
            raise ValueError("Didn't get exactly 1 thing: " + ", ".join(ls))

        path = ls[0]

    with open(path, "rb") as file:
        return pickle.load(file)

In [ ]:
actual_ext = load(dir_actual / "external_data")
actual_unscaled = load(dir_actual / "data-unscaled")
actual_forecast = load(dir_actual / "forecast")

In [ ]:
sim_04_ext = load(dir_sim_04 / "external_data")
sim_04_unscaled = load(dir_sim_04 / "data-unscaled")
sim_04_forecast = load(dir_sim_04 / "forecast")

In [ ]:
sim_15_ext = load(dir_sim_15 / "external_data")
sim_15_unscaled = load(dir_sim_15 / "data-unscaled")
sim_15_forecast = load(dir_sim_15 / "forecast")

### need to check/analyze:
- external data is different for actual vs (all) simulation, but not parts that are put into data-[un]scaled, so probably flow or something.
- data unscaled (specifically non-external so from influx) is different between 04 and 15 sim.

In [ ]:
actual_ext

In [ ]:
# okay it's not flow..
(actual_ext["bafu_flow"] == sim_04_ext["bafu_flow"]).all().all()

In [ ]:
(actual_ext["meteotest"] == sim_04_ext["meteotest"]).all().all()

In [ ]:
actual_ext["meteotest"]

In [ ]:
# the time is the same for both, so nothing weird there
sim_04_ext["meteotest"]

In [ ]:
# some are perfectly identical, some aren't. seems to depend on the column.
(actual_ext["meteotest"] == sim_04_ext["meteotest"]).sum()

In [ ]:
# however once you tolerate minor fluctuations due to precision loss etc. both are identical.
np.allclose(actual_ext["meteotest"].drop(columns=["_time"]).to_numpy(), sim_04_ext["meteotest"].drop(columns=["_time"]).to_numpy())

#### So why is external data different between actual and simulation?

**minor precision differences, probably due to storage and transfer**

### Difference in data from influx between simulations

In [ ]:
# first validate that covariates (coming from external data) aren't any different.
# ps. just now realized again that we don't have any non-linear transformations on water temp
# because that doesn't work autoregressively in darts, ouch that's pretty bad.
sim_04_unscaled

In [ ]:
np.allclose(sim_04_unscaled["future_covariates"].values(), sim_15_unscaled["future_covariates"].values())

In [ ]:
fut_same = (sim_04_unscaled["future_covariates"].to_dataframe() == sim_15_unscaled["future_covariates"].to_dataframe())
fut_same

In [ ]:
# all features are different at just 1h, except for ma3, which is different in 3 hours (makes sense, avg over 3)
fut_same.sum()

In [ ]:
# the three hours that are different are the first 3 (actually just the first (just minutes ago),
# the other 2 are just from ma3 propagating the difference of the first).
_s = fut_same.sum(axis=1)
_s[_s!=5]

In [ ]:
fut_04 = sim_04_unscaled["future_covariates"].to_dataframe()
fut_15 = sim_15_unscaled["future_covariates"].to_dataframe()

In [ ]:
fut_04[between(fut_04, "2026-05-03 15:00:00", "2026-05-03 17:00:01")]

In [ ]:
fut_15[between(fut_15, "2026-05-03 15:00:00", "2026-05-03 17:00:01")]

#### there you have it folks, that first hour is slightly different.

This was made with hourly mean values, both in the model training and the service.
The influxdb sets the label to the right so it seems that between the first run at 17:04 and the second at 17:15,
the data between 16:00 and 17:00 changed slightly (e.g. a datapoint was added) and thus the mean was different.
Next we should test if this can also happen with the FIRST feature modeling.

TODO also check the target, it should have the same behaviour